# GW170817 PE with SHARPy + `mlgw_bns_jax` — Minimal 6-parameter model

This notebook performs Bayesian parameter estimation on GW170817 using:
- **`mlgw_bns_jax`**: JAX-based BNS waveform approximant
- **SHARPy** (original): Sequential Monte Carlo sampler — the waveform template is monkey-patched at runtime
- **Cleaned (deglitched) GWOSC data**: C01/v2 strain with the L1 glitch removed

Minimal 6-parameter model with fixed sky location, polarisation, coalescence phase/time:

| Sampled | Fixed |
|---|---|
| log-distance | RA (NGC 4993) |
| inclination | Dec (NGC 4993) |
| $\mathcal{M}_c$ | polarisation |
| $q$ | $\phi_c$ |
| $\chi_{\mathrm{eff}}$ | $t_c$ |
| $\tilde{\Lambda}$ | |

In [ ]:
from __future__ import annotations

import os, sys, time
from functools import partial

import numpy as np

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

print("JAX devices:", jax.devices())

## Load the waveform model and monkey-patch SHARPy

We replace the original `IMRPhenomD` template in SHARPy with our `mlgw_bns_jax` BNS waveform model **without modifying any SHARPy source file**.

In [ ]:
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_mlgw_predict = load_predict(MODEL_PATH)

# ---- Monkey-patch SHARPy's template -----------------------------------
import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses


def _template_mlgw_bns(params, frequency_array):
    """mlgw_bns_jax waveform, drop-in replacement for SHARPy's template."""
    mc, q = params[6], params[7]
    m1, m2 = McQ2Masses(mc, q)
    total_mass = m1 + m2
    chi1, chi2 = params[9], params[10]
    lambda_1, lambda_2 = params[11], params[12]
    phic = params[4]
    dist_mpc = jnp.exp(params[2])
    inclination = params[3]

    mlgw_params = jnp.array([q, lambda_1, lambda_2, chi1, chi2])
    hp, hc = _mlgw_predict(
        mlgw_params, frequency_array,
        total_mass=total_mass,
        distance_mpc=dist_mpc,
        inclination=inclination,
    )
    phase_factor = jnp.exp(-1j * phic)
    return hp * phase_factor, hc * phase_factor


_gw_mod.template = _template_mlgw_bns

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
from sharpy.smc_functions import run_sharpy
import sharpy.PSDs

print("Model loaded — SHARPy template patched with mlgw_bns_jax.")

## Event parameters and tidal conversion

In [ ]:
def lambda_tilde_to_lambdas(lambda_tilde, m1, m2, delta_lambda_tilde=0.0):
    r"""Convert $(\tilde\Lambda, \delta\tilde\Lambda)$ to individual $(\Lambda_1, \Lambda_2)$."""
    M = m1 + m2
    m1_4, m2_4 = m1**4, m2**4
    eta = (m1 * m2) / M**2
    X = jnp.sqrt(1.0 - 4.0 * eta)
    c1 = (16.0 / 13.0) * (m1 + 12.0 * m2) * m1_4 / M**5
    c2 = (16.0 / 13.0) * (m2 + 12.0 * m1) * m2_4 / M**5
    a = 1690.0 * eta / 1319.0 - 4843.0 / 1319.0
    b = 6162.0 * X / 1319.0
    d1 = (a + b) * m1_4 / M**4
    d2 = (-a + b) * m2_4 / M**4
    det = c1 * d2 - c2 * d1
    return (d2 * lambda_tilde - c2 * delta_lambda_tilde) / det, \
           (c1 * delta_lambda_tilde - d1 * lambda_tilde) / det


TRIGGER_TIME = 1187008882.43
SEGMENT_DURATION = 4.0
SAMPLING_RATE = 4096
F_LOWER = 20.0
F_UPPER = 2000.0
DATA_START_GPS = 1187008867
DATA_DURATION = 32

FIXED_RA = 3.44616        # NGC 4993
FIXED_DEC = -0.408084
FIXED_POL = 0.0
FIXED_PHIC = 0.0
FIXED_TC = 0.0

DATA_DIR = "gw170817_data"
OUTDIR = "outdir_GW170817_sharpy_minimal"
LABEL = "GW170817_sharpy_minimal"
os.makedirs(OUTDIR, exist_ok=True)

## Load cleaned data and build detector network

SHARPy's `load_data` parses the filename to extract GPS start time and duration (`DET-FRAMETYPE-START-DURATION.txt`).
We create symlinks from the cleaned files to LIGO-convention names.

In [ ]:
cleaned_sources = {
    "H1": os.path.join(DATA_DIR, "H1_cleaned.txt"),
    "L1": os.path.join(DATA_DIR, "L1_cleaned.txt"),
    "V1": os.path.join(DATA_DIR, "V1_cleaned.txt"),
}
cleaned_ligo = {
    "H1": os.path.join(DATA_DIR, f"H-H1_CLEANED_C01-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_CLEANED_C01-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_CLEANED_C01-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det in ["H1", "L1", "V1"]:
    src = os.path.abspath(cleaned_sources[det])
    dst = cleaned_ligo[det]
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f"{det}: {os.path.basename(dst)} -> {os.path.basename(src)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=cleaned_ligo[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE,
        f_lower=F_LOWER, f_upper=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print("\nBuilding GW network...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")

## Define reduced likelihood and priors

In [ ]:
batched_detector = gw_network.batched_detector
log_likelihood_full = partial(log_likelihood_det, detector_list=batched_detector)


def log_likelihood_reduced(params_6):
    """Expand 6 sampled params -> full 13-param SHARPy vector."""
    logdist, incl, mc, q, chi_eff, lambda_tilde = (
        params_6[0], params_6[1], params_6[2],
        params_6[3], params_6[4], params_6[5],
    )
    m1, m2 = McQ2Masses(mc, q)
    l1, l2 = lambda_tilde_to_lambdas(lambda_tilde, m1, m2)
    l1, l2 = jnp.clip(l1, 0.0), jnp.clip(l2, 0.0)

    params_13 = jnp.array([
        FIXED_RA, FIXED_DEC, logdist, incl, FIXED_PHIC,
        FIXED_POL, mc, q, FIXED_TC,
        chi_eff, chi_eff,  # chi1 = chi2 = chi_eff
        l1, l2,
    ])
    return log_likelihood_full(params_13)


prior_bounds = jnp.array([
    [jnp.log(10.0), jnp.log(100.0)],  # logdistance
    [0.0, jnp.pi],                     # inclination
    [1.18, 1.21],                       # mc
    [0.5, 1.0],                         # q
    [-0.05, 0.05],                      # chi_eff
    [0.0, 5000.0],                      # lambda_tilde
])

boundary_conditions = jnp.array([0, 0, 0, 0, 0, 0])

parameter_names = ["logdistance", "theta_jn", "mc", "q", "chi_eff", "lambda_tilde"]


def prior(params):
    return 0.0


print(f"Sampling {len(parameter_names)} parameters: {parameter_names}")

## Run the SMC sampler

In [ ]:
N_PARTICLES = 500
STEP_SIZE = 0.3
ALPHA = 0.95
SEED = 42

print(f"Starting SHARPy SMC with {N_PARTICLES} particles...")
start = time.time()

result_dict = run_sharpy(
    log_likelihood_reduced, prior,
    prior_bounds, boundary_conditions,
    ALPHA, N_PARTICLES, STEP_SIZE,
    jax.random.PRNGKey(SEED),
    folder=OUTDIR, label=LABEL,
)

dt = time.time() - start
samples = result_dict["posterior_samples"]
logZ, dlogZ = result_dict["logZ"], result_dict["dlogZ"]
print(f"\nDone in {dt:.1f} s — log Z = {logZ:.2f} ± {dlogZ:.2f}")

## Corner plot

In [ ]:
from corner import corner

fig = corner(
    np.array(samples), show_titles=True,
    labels=parameter_names, title_kwargs={"fontsize": 12},
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner.png")
fig.savefig(plot_path)
print(f"Saved to {plot_path}")
fig